# A04 — Training UNetBaseline con Perceptual Loss (Mitigazione)

Addestra `UNetBaseline` con la loss **L1 + SSIM + VGGPerceptual** al posto
della sola L1 + SSIM. **Architettura identica alla baseline** — solo la loss cambia.
Questo isola l'effetto della supervisione percettiva dal cambio architetturale (CBAM).

| | |
|---|---|
| Modello | UNetBaseline (base_channels=32) — stessa architettura della baseline |
| Loss | CombinedPerceptualLoss: L1 + (1−SSIM) + 0.05 · VGGPerceptual |
| Dataset | LOL-v2 Real_captured (stesso split seed=42 della baseline) |
| Hardware | Kaggle T4 (16 GB VRAM) |
| Checkpoint | `checkpoints/perceptual/best.pt` |

**Differenze rispetto a baseline (T07):**
- `criterion = CombinedPerceptualLoss(alpha=0.8, lambda_perc=0.05)`
- `experiment_name = "perceptual"`
- Tutto il resto identico → confronto diretto con baseline e attention

In [ ]:
# ── Dipendenze ────────────────────────────────────────────────────────────────
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs, "-q"])

for pkg, imp in [("piq>=0.8.0", "piq"), ("torchinfo>=1.8.0", "torchinfo")]:
    try:
        __import__(imp)
    except ImportError:
        _pip(pkg)

print("Dipendenze OK")

In [ ]:
import os, sys
from pathlib import Path
import torch
from torch.utils.data import DataLoader, Subset

ON_KAGGLE = Path("/kaggle/working").exists()

if ON_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/input/datasets/mattiacagnetta/low-light-src")
    OUTPUT_ROOT  = Path("/kaggle/working")
    LOLV2_ROOT   = Path("/kaggle/input/datasets/mattiacagnetta/lol-v2/LOL-v2")
else:
    PROJECT_ROOT = Path("..").resolve()
    OUTPUT_ROOT  = PROJECT_ROOT
    LOLV2_ROOT   = PROJECT_ROOT / "data" / "LOL-v2"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"ON_KAGGLE    : {ON_KAGGLE}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
TRAIN_LOW    = LOLV2_ROOT / "Real_captured" / "Train" / "Low"
TRAIN_NORMAL = LOLV2_ROOT / "Real_captured" / "Train" / "Normal"
TEST_LOW     = LOLV2_ROOT / "Real_captured" / "Test"  / "Low"
TEST_NORMAL  = LOLV2_ROOT / "Real_captured" / "Test"  / "Normal"

SPLITS_DIR = OUTPUT_ROOT / "data" / "splits"
CKPT_DIR   = OUTPUT_ROOT / "checkpoints"
LOG_DIR    = OUTPUT_ROOT / "runs"

BATCH_SIZE  = 8
NUM_WORKERS = 2
IMG_SIZE    = 256

for p in [TRAIN_LOW, TRAIN_NORMAL, TEST_LOW, TEST_NORMAL]:
    print(f"  {'✓' if p.exists() else '✗ NON TROVATO'}  {p}")

In [ ]:
# ── Split identici a baseline e attention (seed=42, val=10%) ─────────────────
from src.data.dataset import PairedImageDataset
from src.data.splits  import create_and_save_train_val_split, save_test_split

lolv2_key = lambda s: s.lstrip("abcdefghijklmnopqrstuvwxyz")

full_train_ds = PairedImageDataset(TRAIN_LOW, TRAIN_NORMAL, key_fn=lolv2_key)
test_ds_raw   = PairedImageDataset(TEST_LOW,  TEST_NORMAL,  key_fn=lolv2_key)

print(f"Train: {len(full_train_ds)}  |  Test: {len(test_ds_raw)}")

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
splits = create_and_save_train_val_split(
    full_train_ds.stems, splits_dir=SPLITS_DIR,
    split_name="lolv2_real", val_fraction=0.10, seed=42,
)
save_test_split(test_ds_raw.stems, SPLITS_DIR / "lolv2_real_test.txt")

train_stems, val_stems = splits["train"], splits["val"]
print(f"Train split: {len(train_stems)}  |  Val split: {len(val_stems)}")

In [ ]:
# ── Dataset e DataLoader ──────────────────────────────────────────────────────
from src.data.transforms import get_preprocessing_transform, get_paired_augmentation

preprocess = get_preprocessing_transform(IMG_SIZE)
augment    = get_paired_augmentation(IMG_SIZE)

train_ds_full = PairedImageDataset(TRAIN_LOW, TRAIN_NORMAL, key_fn=lolv2_key,
                                   paired_transform=augment, transform=preprocess)
eval_ds_full  = PairedImageDataset(TRAIN_LOW, TRAIN_NORMAL, key_fn=lolv2_key,
                                   transform=preprocess)

train_set, val_set = set(train_stems), set(val_stems)
train_indices = [i for i, s in enumerate(train_ds_full.stems) if s in train_set]
val_indices   = [i for i, s in enumerate(eval_ds_full.stems)  if s in val_set]

train_loader = DataLoader(
    Subset(train_ds_full, train_indices),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0), drop_last=True,
)
val_loader = DataLoader(
    Subset(eval_ds_full, val_indices),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)

print(f"Train: {len(train_indices)} sample  |  Val: {len(val_indices)} sample")

In [ ]:
# ── Modello, Loss, Config ─────────────────────────────────────────────────────
# UNICA DIFFERENZA rispetto a T07:
#   - criterion = CombinedPerceptualLoss (L1 + SSIM + VGG)
#   - experiment_name = "perceptual"
# Tutto il resto è identico per un confronto equo.

from src.models  import UNetBaseline
from src.losses  import CombinedPerceptualLoss
from src.train   import TrainConfig, Trainer

model     = UNetBaseline(base_channels=32)
criterion = CombinedPerceptualLoss(alpha=0.8, lambda_perc=0.05)

config = TrainConfig(
    optimizer_name          = "adamw",
    lr                      = 1e-4,
    weight_decay            = 1e-4,
    grad_clip_norm          = 1.0,
    epochs                  = 100,
    scheduler_name          = "cosine",
    early_stopping_patience = 15,
    val_every_n_epochs      = 1,
    log_every_n_epochs      = 1,
    n_visual_samples        = 4,
    amp                     = True,
    device                  = "auto",
    seed                    = 42,
    checkpoint_dir          = str(CKPT_DIR),
    log_dir                 = str(LOG_DIR),
    experiment_name         = "perceptual",   # <── cartella separata
)

print(f"Modello     : {model.__class__.__name__}")
print(f"Parametri   : {sum(p.numel() for p in model.parameters()):,}")
print(f"Loss        : {criterion}")
print(f"Experiment  : {config.experiment_name}")

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
trainer = Trainer(model, train_loader, val_loader, criterion, config)
trainer.fit()

In [ ]:
# ── Riepilogo e confronto ─────────────────────────────────────────────────────
import json
from dataclasses import asdict

best_ckpt = trainer.ckpt_dir / "best.pt"

# Ricarica il best checkpoint per valutazione finale
ckpt = torch.load(best_ckpt, map_location=trainer.device, weights_only=False)
trainer.model.load_state_dict(ckpt["model"])
final_metrics = trainer.val_epoch()

print("=" * 65)
print("RIEPILOGO — UNetBaseline + Perceptual Loss")
print("=" * 65)
print(f"Epoche completate : {trainer.epoch}")
print(f"Best val_loss     : {trainer.best_score:.6f}")
print(f"PSNR (val)        : {final_metrics['psnr']:.2f} dB")
print(f"SSIM (val)        : {final_metrics['ssim']:.4f}")
print()

# Confronto con baseline e attention (se disponibili)
for exp in ["baseline", "attention"]:
    s_path = CKPT_DIR / exp / "training_summary.json"
    if s_path.exists():
        s = json.loads(s_path.read_text(encoding="utf-8"))
        print(f"vs {exp:12}: PSNR {s['val_psnr_db']:.2f} dB  SSIM {s['val_ssim']:.4f}  "
              f"→ ΔPSNR={final_metrics['psnr']-s['val_psnr_db']:+.2f}  "
              f"ΔSSIM={final_metrics['ssim']-s['val_ssim']:+.4f}")

# Salva summary JSON
summary = {
    "experiment"    : config.experiment_name,
    "model"         : model.__class__.__name__,
    "loss"          : str(criterion),
    "lambda_perc"   : criterion.lambda_perc,
    "epochs_done"   : trainer.epoch,
    "best_val_loss" : trainer.best_score,
    "val_psnr_db"   : final_metrics["psnr"],
    "val_ssim"      : final_metrics["ssim"],
    "hardware"      : torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "config"        : asdict(config),
}
(trainer.ckpt_dir / "training_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
print(f"\nSummary salvato: {trainer.ckpt_dir / 'training_summary.json'}")

# Download su Kaggle
if ON_KAGGLE:
    import shutil
    from IPython.display import FileLink, display
    shutil.make_archive(
        "/kaggle/working/perceptual_checkpoint", "zip",
        root_dir=str(trainer.ckpt_dir.parent),
        base_dir="perceptual",
    )
    print("\nZip pronto:")
    display(FileLink("perceptual_checkpoint.zip"))